# WLASL Sign Language Recognition with CNN-Mamba

This notebook implements a **MobileNetV3-Large + Selective State Space Model (Mamba)** architecture for real-time word-level American Sign Language (ASL) recognition on the WLASL dataset.

**Architecture:**
- **Spatial Encoder:** MobileNetV3-Large (ImageNet pretrained)
- **Temporal Decoder:** 2-layer selective SSM (Mamba)
- **Input:** 25 frames × 224 × 224 RGB (1 second at 25 fps)
- **Output:** 300 ASL gloss classes (WLASL300 subset)

**Key Properties:**
- Linear-time temporal complexity: O(n)
- Fixed memory state during inference (no KV cache growth)
- ~6M parameters, suitable for edge deployment

## 1. Setup and Imports

In [ ]:
# Install required packages (run once)
!pip install -q opencv-python scikit-learn seaborn

import os
import sys
import json
import time
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_large

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

Dataset paths use a dual-source fallback system: primary videos from `wlasl-processed` with automatic fallback to `wlasl2000-resized` for missing files.

In [ ]:
# Configuration
class Config:
    # Dataset
    NUM_CLASSES = 300          # WLASL300 (or 100, 1000, 2000)
    NUM_FRAMES = 25            # Frames per video (1 second at 25 fps)
    FRAME_SIZE = 224           # Spatial resolution
    
    # Model
    D_MODEL = 256              # Mamba hidden dimension
    D_STATE = 16               # SSM state dimension
    N_MAMBA_LAYERS = 2         # Number of Mamba blocks
    DROPOUT = 0.5
    
    # Training
    BATCH_SIZE = 32
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE = 10              # Early stopping patience
    
    # Paths (Kaggle dual dataset fallback)
    JSON_PATH = '/kaggle/input/wlasl-processed/WLASL_v0.3.json'
    VIDEO_ROOT = '/kaggle/input/wlasl-processed/videos'
    VIDEO_ROOT_BACKUP = '/kaggle/input/wlasl2000-resized/wlasl-complete/videos'
    CHECKPOINT_DIR = './checkpoints'
    
    # Reproducibility
    SEED = 42

# Create checkpoint directory
os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

# Set random seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(Config.SEED)
print("Configuration loaded successfully!")

## 3. Data Exploration

In [ ]:
# Load WLASL annotations
def load_wlasl_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    return data

data = load_wlasl_data(Config.JSON_PATH)
print(f"Total glosses in full dataset: {len(data)}")

# Get statistics for top-k classes
def analyze_subset(data, k=300):
    gloss_counts = [(entry['gloss'], len(entry['instances'])) for entry in data]
    gloss_counts.sort(key=lambda x: x[1], reverse=True)
    top_k = gloss_counts[:k]
    
    # Statistics
    subset_stats = {'train': 0, 'val': 0, 'test': 0}
    counts = []
    
    for gloss, _ in top_k:
        entry = next(e for e in data if e['gloss'] == gloss)
        class_count = {'train': 0, 'val': 0, 'test': 0, 'total': 0}
        for inst in entry['instances']:
            split = inst['split']
            subset_stats[split] += 1
            class_count[split] += 1
            class_count['total'] += 1
        counts.append(class_count['total'])
    
    total = sum(subset_stats.values())
    print(f"\nWLASL{k} Statistics:")
    print(f"  Total videos: {total}")
    print(f"  Train: {subset_stats['train']} ({subset_stats['train']/total*100:.1f}%)")
    print(f"  Val: {subset_stats['val']} ({subset_stats['val']/total*100:.1f}%)")
    print(f"  Test: {subset_stats['test']} ({subset_stats['test']/total*100:.1f}%)")
    print(f"  Classes: {k}")
    print(f"  Instances per class: min={min(counts)}, max={max(counts)}, mean={np.mean(counts):.1f}")
    
    return top_k

top_300 = analyze_subset(data, k=Config.NUM_CLASSES)

# Visualize class distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
counts = [c[1] for c in top_300]
plt.hist(counts, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Instances per Class')
plt.ylabel('Number of Classes')
plt.title('WLASL300: Instances per Class')
plt.axvline(np.mean(counts), color='red', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
plt.legend()

plt.subplot(1, 2, 2)
names = [c[0] for c in top_300[:20]]
values = [c[1] for c in top_300[:20]]
plt.barh(range(len(names)), values, color='steelblue')
plt.yticks(range(len(names)), names)
plt.xlabel('Number of Instances')
plt.title('Top 20 Most Frequent Classes')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nClass distribution plot saved!")

## 4. Data Loading

In [ ]:
class WLASLDataset(Dataset):
    """WLASL Dataset for video sign language recognition."""
    
    def __init__(self, json_path, video_root, split='train', num_classes=300,
                 num_frames=25, frame_size=224, transform=None, video_root_backup=None):
        self.video_root = video_root
        self.video_root_backup = video_root_backup
        self.split = split
        self.num_frames = num_frames
        self.frame_size = frame_size
        self.transform = transform
        
        # Load annotations
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        # Select top-k classes
        gloss_counts = [(entry['gloss'], len(entry['instances'])) for entry in data]
        gloss_counts.sort(key=lambda x: x[1], reverse=True)
        self.top_k_glosses = [g[0] for g in gloss_counts[:num_classes]]
        
        # Create class to index mapping
        self.class_to_idx = {gloss: idx for idx, gloss in enumerate(self.top_k_glosses)}
        
        # Build dataset samples
        self.samples = []
        for entry in data:
            if entry['gloss'] not in self.top_k_glosses:
                continue
            
            gloss = entry['gloss']
            label = self.class_to_idx[gloss]
            
            for inst in entry['instances']:
                inst_split = inst['split']
                if split == 'train' and inst_split not in ['train', 'val']:
                    continue
                elif split in ['val', 'test'] and inst_split != split:
                    continue
                
                video_id = inst['video_id']
                video_path = os.path.join(video_root, f"{video_id}.mp4")
                
                if not os.path.exists(video_path) and video_root_backup is not None:
                    video_path = os.path.join(video_root_backup, f"{video_id}.mp4")
                
                if not os.path.exists(video_path):
                    continue
                
                self.samples.append({
                    'video_id': video_id,
                    'video_path': video_path,
                    'label': label,
                    'gloss': gloss
                })
        
        print(f"WLASL{num_classes} {split}: {len(self.samples)} samples loaded")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        frames = self.load_video(sample['video_path'])
        
        if self.transform:
            frames = self.transform(frames)
        
        return frames, sample['label'], sample['video_id']
    
    def load_video(self, video_path):
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            return torch.zeros(3, self.num_frames, self.frame_size, self.frame_size)
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            return torch.zeros(3, self.num_frames, self.frame_size, self.frame_size)
        
        # Sample frame indices uniformly
        indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
        
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            
            if not ret:
                if len(frames) > 0:
                    frames.append(frames[-1].copy())
                else:
                    frames.append(np.zeros((self.frame_size, self.frame_size, 3), dtype=np.uint8))
                continue
            
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (self.frame_size, self.frame_size))
            frame = frame.astype(np.float32) / 255.0
            frames.append(frame)
        
        cap.release()
        
        frames = np.stack(frames, axis=0)
        frames = np.transpose(frames, (3, 0, 1, 2))
        return torch.from_numpy(frames).float()


class RandomHorizontalFlipVideo:
    def __init__(self, p=0.5):
        self.p = p
    
    def __call__(self, frames):
        if random.random() < self.p:
            frames = torch.flip(frames, dims=[-1])
        return frames

class NormalizeVideo:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean).view(3, 1, 1, 1)
        self.std = torch.tensor(std).view(3, 1, 1, 1)
    
    def __call__(self, frames):
        self.mean = self.mean.to(frames.device)
        self.std = self.std.to(frames.device)
        return (frames - self.mean) / self.std

def get_data_loaders(json_path, video_root, video_root_backup=None, num_classes=300,
                     num_frames=25, frame_size=224, batch_size=32, num_workers=4):
    train_transform = transforms.Compose([
        RandomHorizontalFlipVideo(p=0.5),
        NormalizeVideo(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        NormalizeVideo(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = WLASLDataset(
        json_path=json_path,
        video_root=video_root,
        split='train',
        num_classes=num_classes,
        num_frames=num_frames,
        frame_size=frame_size,
        transform=train_transform,
        video_root_backup=video_root_backup
    )
    
    val_dataset = WLASLDataset(
        json_path=json_path,
        video_root=video_root,
        split='val',
        num_classes=num_classes,
        num_frames=num_frames,
        frame_size=frame_size,
        transform=val_transform,
        video_root_backup=video_root_backup
    )
    
    test_dataset = WLASLDataset(
        json_path=json_path,
        video_root=video_root,
        split='test',
        num_classes=num_classes,
        num_frames=num_frames,
        frame_size=frame_size,
        transform=val_transform,
        video_root_backup=video_root_backup
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )
    
    print(f"\nData loaders created:")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")
    
    return {
        'train': train_loader,
        'val': val_loader,
        'test': test_loader
    }


# Create data loaders
JSON_PATH = Config.JSON_PATH
VIDEO_ROOT = Config.VIDEO_ROOT
VIDEO_ROOT_BACKUP = Config.VIDEO_ROOT_BACKUP
NUM_CLASSES = Config.NUM_CLASSES
NUM_FRAMES = Config.NUM_FRAMES
FRAME_SIZE = Config.FRAME_SIZE
BATCH_SIZE = Config.BATCH_SIZE

loaders = get_data_loaders(
    json_path=JSON_PATH,
    video_root=VIDEO_ROOT,
    video_root_backup=VIDEO_ROOT_BACKUP,
    num_classes=NUM_CLASSES,
    num_frames=NUM_FRAMES,
    frame_size=FRAME_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=4
)

train_loader = loaders['train']
val_loader = loaders['val']
test_loader = loaders['test']

## 5. Model Definition

In [ ]:
class SelectiveSSM(nn.Module):
    """Simplified Selective State Space Model (Mamba) in pure PyTorch."""
    
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner, out_channels=self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1,
            groups=self.d_inner, bias=True
        )
        
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)
        
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
    
    def forward(self, x):
        B, L, D = x.shape
        
        x_and_res = self.in_proj(x)
        x_ssm, res = x_and_res.split([self.d_inner, self.d_inner], dim=-1)
        
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :L]
        x_conv = x_conv.transpose(1, 2)
        x_conv = F.silu(x_conv)
        
        y = self.ssm(x_conv)
        y = y * F.silu(res)
        
        return self.out_proj(y)
    
    def ssm(self, x):
        B, L, _ = x.shape
        
        x_dbl = self.x_proj(x)
        delta, B_ssm, C = x_dbl.split([1, self.d_state, self.d_state], dim=-1)
        
        delta = F.softplus(self.dt_proj(delta))
        A = -torch.exp(self.A_log.float())
        
        return self.selective_scan(x, delta, A, B_ssm, C)
    
    def selective_scan(self, x, delta, A, B, C):
        B, L, d_in = x.shape
        d_state = A.size(1)
        
        h = torch.zeros(B, d_in, d_state, device=x.device, dtype=x.dtype)
        ys = []
        
        for t in range(L):
            delta_t = delta[:, t, :]
            A_bar = torch.exp(delta_t.unsqueeze(-1) * A.unsqueeze(0))
            B_bar = delta_t.unsqueeze(-1) * B[:, t, :].unsqueeze(1)
            
            h = A_bar * h + B_bar * x[:, t, :].unsqueeze(-1)
            y_t = torch.sum(C[:, t, :].unsqueeze(1) * h, dim=-1)
            ys.append(y_t)
        
        y = torch.stack(ys, dim=1)
        y = y + x * self.D.unsqueeze(0).unsqueeze(0)
        
        return y


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mixer = SelectiveSSM(d_model, d_state, d_conv, expand)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        residual = x
        x = self.norm(x)
        x = self.mixer(x)
        x = self.dropout(x)
        return x + residual


class MobileNetV3_Mamba(nn.Module):
    """CNN-Mamba hybrid for sign language recognition."""
    
    def __init__(self, num_classes=300, d_model=256, n_mamba_layers=2,
                 d_state=16, dropout=0.5, pretrained=True):
        super().__init__()
        
        # Spatial encoder
        mobilenet = mobilenet_v3_large(pretrained=pretrained)
        self.spatial_encoder = nn.Sequential(*list(mobilenet.features.children()))
        self.spatial_pool = nn.AdaptiveAvgPool2d(1)
        self.spatial_dim = 960
        
        # Projection
        self.input_proj = nn.Linear(self.spatial_dim, d_model)
        
        # Temporal decoder
        self.temporal_decoder = nn.ModuleList([
            MambaBlock(d_model, d_state=d_state, dropout=dropout)
            for _ in range(n_mamba_layers)
        ])
        
        # Classification head
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def extract_frame_features(self, frames):
        B, C, T, H, W = frames.shape
        frames_flat = frames.permute(0, 2, 1, 3, 4).contiguous().view(B * T, C, H, W)
        
        features = self.spatial_encoder(frames_flat)
        features = self.spatial_pool(features)
        features = features.view(B, T, self.spatial_dim)
        
        return features
    
    def forward(self, frames):
        x = self.extract_frame_features(frames)
        x = self.input_proj(x)
        
        for block in self.temporal_decoder:
            x = block(x)
        
        x = x.transpose(1, 2)
        x = self.temporal_pool(x).squeeze(-1)
        
        return self.classifier(x)

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = MobileNetV3_Mamba(
    num_classes=Config.NUM_CLASSES,
    d_model=Config.D_MODEL,
    n_mamba_layers=Config.N_MAMBA_LAYERS,
    d_state=Config.D_STATE,
    dropout=Config.DROPOUT,
    pretrained=True
)

# Use DataParallel if multiple GPUs available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Summary:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size (FP32): {total_params * 4 / (1024**2):.2f} MB")

# Test forward pass
dummy_input = torch.randn(2, 3, Config.NUM_FRAMES, Config.FRAME_SIZE, Config.FRAME_SIZE).to(device)
output = model(dummy_input)
print(f"\nTest forward pass:")
print(f"  Input shape: {dummy_input.shape}")
print(f"  Output shape: {output.shape}")

## 6. Training

In [ ]:
# Setup training
criterion = nn.CrossEntropyLoss()

# Separate learning rates for pretrained and new layers
pretrained_params = []
new_params = []
for name, param in model.named_parameters():
    if 'spatial_encoder' in name:
        pretrained_params.append(param)
    else:
        new_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': pretrained_params, 'lr': Config.LEARNING_RATE * 0.1},
    {'params': new_params, 'lr': Config.LEARNING_RATE}
], weight_decay=Config.WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.3, verbose=True
)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_top5': [],
    'val_top10': []
}

best_val_acc = 0.0
patience_counter = 0

def accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res

# Training loop
for epoch in range(1, Config.NUM_EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{Config.NUM_EPOCHS}")
    print(f"{'='*60}")
    
    # Training
    model.train()
    train_loss = 0.0
    train_top1 = 0.0
    train_top5 = 0.0
    
    for batch_idx, (frames, labels, _) in enumerate(train_loader):
        frames = frames.to(device)
        labels = labels.to(device)
        
        outputs = model(frames)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        acc1, acc5 = accuracy(outputs, labels, topk=(1, 5))
        train_loss += loss.item()
        train_top1 += acc1.item()
        train_top5 += acc5.item()
        
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f} "
                  f"Top-1: {acc1.item():.2f}%")
    
    train_loss /= len(train_loader)
    train_top1 /= len(train_loader)
    train_top5 /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_top1 = 0.0
    val_top5 = 0.0
    val_top10 = 0.0
    
    with torch.no_grad():
        for frames, labels, _ in val_loader:
            frames = frames.to(device)
            labels = labels.to(device)
            
            outputs = model(frames)
            loss = criterion(outputs, labels)
            
            acc1, acc5, acc10 = accuracy(outputs, labels, topk=(1, 5, 10))
            val_loss += loss.item()
            val_top1 += acc1.item()
            val_top5 += acc5.item()
            val_top10 += acc10.item()
    
    val_loss /= len(val_loader)
    val_top1 /= len(val_loader)
    val_top5 /= len(val_loader)
    val_top10 /= len(val_loader)
    
    # Update history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_top1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_top1)
    history['val_top5'].append(val_top5)
    history['val_top10'].append(val_top10)
    
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Top-1: {train_top1:.2f}% | Top-5: {train_top5:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Top-1: {val_top1:.2f}% | Top-5: {val_top5:.2f}% | Top-10: {val_top10:.2f}%")
    
    # Save best model
    if val_top1 > best_val_acc:
        best_val_acc = val_top1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_top1,
            'history': history
        }, f'{Config.CHECKPOINT_DIR}/best_model.pth')
        print(f"  ✓ New best model saved (Val Top-1: {val_top1:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= Config.PATIENCE:
        print(f"\nEarly stopping triggered after {epoch} epochs")
        break
    
    # Learning rate scheduling
    scheduler.step(val_loss)

print(f"\nTraining completed! Best validation Top-1: {best_val_acc:.2f}%")

## 7. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'], label='Val Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Loss over Epochs')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['train_acc'], label='Train Top-1')
axes[0, 1].plot(history['val_acc'], label='Val Top-1')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Top-1 Accuracy over Epochs')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['val_top5'], label='Val Top-5', color='orange')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy (%)')
axes[1, 0].set_title('Validation Top-5 Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history['val_top10'], label='Val Top-10', color='green')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].set_title('Validation Top-10 Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training history plot saved!")

## 8. Test Set Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(f'{Config.CHECKPOINT_DIR}/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']} "
      f"(Val Top-1: {checkpoint['val_acc']:.2f}%)")

# Evaluate on test set
test_top1 = 0.0
test_top5 = 0.0
test_top10 = 0.0

all_preds = []
all_labels = []
inference_times = []

with torch.no_grad():
    for frames, labels, _ in test_loader:
        frames = frames.to(device)
        labels = labels.to(device)
        
        # Measure inference time
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        
        outputs = model(frames)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.time()
        
        batch_time = (end - start) / frames.size(0)
        inference_times.append(batch_time)
        
        # Metrics
        acc1, acc5, acc10 = accuracy(outputs, labels, topk=(1, 5, 10))
        test_top1 += acc1.item()
        test_top5 += acc5.item()
        test_top10 += acc10.item()
        
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_top1 /= len(test_loader)
test_top5 /= len(test_loader)
test_top10 /= len(test_loader)
avg_inference_time = np.mean(inference_times) * 1000

# Per-class accuracy
class_correct = defaultdict(int)
class_total = defaultdict(int)
for pred, label in zip(all_preds, all_labels):
    class_total[label] += 1
    if pred == label:
        class_correct[label] += 1

per_class_acc = np.mean([class_correct[c] / class_total[c] for c in class_total]) * 100

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"Top-1 Accuracy: {test_top1:.2f}%")
print(f"Top-5 Accuracy: {test_top5:.2f}%")
print(f"Top-10 Accuracy: {test_top10:.2f}%")
print(f"Per-class Accuracy: {per_class_acc:.2f}%")
print(f"Avg Inference Time: {avg_inference_time:.2f} ms/video")
print(f"Throughput: {1000/avg_inference_time:.1f} videos/sec")
print(f"Model Parameters: {total_params:,}")
print(f"Model Size (FP32): {total_params * 4 / (1024**2):.2f} MB")
print("=" * 60)

# Save results
results = {
    'top1': test_top1,
    'top5': test_top5,
    'top10': test_top10,
    'per_class_acc': per_class_acc,
    'inference_time_ms': avg_inference_time,
    'throughput': 1000/avg_inference_time,
    'num_parameters': total_params,
    'model_size_mb': total_params * 4 / (1024**2)
}

import json
with open('test_results.json', 'w') as f:
    json.dump({k: float(v) if isinstance(v, (np.floating, np.integer)) else v 
               for k, v in results.items()}, f, indent=2)
print("\nResults saved to test_results.json")

In [ ]:
# CPU Speed Test for Edge Deployment
print("\n" + "=" * 60)
print("CPU INFERENCE SPEED TEST (Edge Deployment Simulation)")
print("=" * 60)

class Trainer:
    def __init__(self, model):
        self.model = model
    
    def evaluate_cpu_speed(self, dataloader, num_runs=100):
        self.model.eval()
        self.model = self.model.cpu()
        
        # Get a sample input
        sample_input = None
        for frames, _, _ in dataloader:
            sample_input = frames[:1].cpu()
            break
        
        if sample_input is None:
            return {'avg_ms': 0.0, 'std_ms': 0.0}
        
        # Warmup
        with torch.no_grad():
            for _ in range(10):
                _ = self.model(sample_input)
        
        # Benchmark
        times = []
        for _ in range(num_runs):
            start = time.time()
            with torch.no_grad():
                _ = self.model(sample_input)
            end = time.time()
            times.append((end - start) * 1000)
        
        return {
            'avg_ms': np.mean(times),
            'std_ms': np.std(times)
        }

trainer = Trainer(model)
results_cpu = trainer.evaluate_cpu_speed(loaders['test'], num_runs=100)

print(f"\nCPU Inference: {results_cpu['avg_ms']:.2f} ± {results_cpu['std_ms']:.2f} ms")
print(f"CPU Throughput: {1000/results_cpu['avg_ms']:.1f} videos/sec")

# Also update saved results with CPU metrics
results['cpu_inference_ms'] = results_cpu['avg_ms']
results['cpu_throughput'] = 1000 / results_cpu['avg_ms']
with open('test_results.json', 'w') as f:
    json.dump({k: float(v) if isinstance(v, (np.floating, np.integer)) else v 
               for k, v in results.items()}, f, indent=2)
print("Updated results with CPU metrics")

## 9. Confusion Matrix and Analysis

In [ ]:
# Plot confusion matrix for top 20 classes
from sklearn.metrics import confusion_matrix

class_counts = Counter(all_labels)
top_20_classes = [c for c, _ in class_counts.most_common(20)]

mask = np.isin(all_labels, top_20_classes)
y_true_subset = np.array(all_labels)[mask]
y_pred_subset = np.array(all_preds)[mask]

cm = confusion_matrix(y_true_subset, y_pred_subset, labels=top_20_classes)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(14, 12))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=top_20_classes, yticklabels=top_20_classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Normalized Confusion Matrix (Top 20 Classes)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved!")

# Identify most confused class pairs
confusions = []
for i in range(len(top_20_classes)):
    for j in range(len(top_20_classes)):
        if i != j and cm[i, j] > 0:
            confusions.append((top_20_classes[i], top_20_classes[j], cm[i, j]))

confusions.sort(key=lambda x: x[2], reverse=True)
print("\nTop 10 most confused pairs:")
for true_class, pred_class, count in confusions[:10]:
    print(f"  True: {true_class}, Pred: {pred_class}, Count: {count}")

## 10. Failure Case Analysis

In [ ]:
# Analyze failure cases
failures = [(i, true, pred) for i, (true, pred) in enumerate(zip(all_labels, all_preds)) if true != pred]
print(f"Total failures: {len(failures)} out of {len(all_labels)} ({len(failures)/len(all_labels)*100:.1f}%)")

# Most commonly misclassified classes
misclassified_counts = Counter([true for _, true, _ in failures])
print("\nMost commonly misclassified classes:")
for class_idx, count in misclassified_counts.most_common(10):
    class_name = top_300[class_idx][0] if class_idx < len(top_300) else f"Class_{class_idx}"
    total = class_counts[class_idx]
    print(f"  {class_name}: {count}/{total} misclassified ({count/total*100:.1f}%)")

# Most common wrong predictions
wrong_pred_counts = Counter([pred for _, _, pred in failures])
print("\nMost common wrong predictions:")
for class_idx, count in wrong_pred_counts.most_common(10):
    class_name = top_300[class_idx][0] if class_idx < len(top_300) else f"Class_{class_idx}"
    print(f"  {class_name}: predicted {count} times incorrectly")

## 11. Export Model for Edge Deployment

In [ ]:
# Export to TorchScript for deployment
model.eval()

# Trace the model
# Note: For CPU export, ensure model is on CPU before tracing
model = model.to(device)
example_input = torch.randn(1, 3, Config.NUM_FRAMES, Config.FRAME_SIZE, Config.FRAME_SIZE).to(device)

try:
    traced_model = torch.jit.trace(model, example_input)
    traced_model.save('mobilenetv3_mamba_wlasl300.pt')
    print("Model exported to TorchScript: mobilenetv3_mamba_wlasl300.pt")
    
    # Verify
    loaded_model = torch.jit.load('mobilenetv3_mamba_wlasl300.pt')
    with torch.no_grad():
        output = loaded_model(example_input)
    print(f"Verification successful! Output shape: {output.shape}")
except Exception as e:
    print(f"TorchScript export failed: {e}")
    print("Saving state dict instead...")
    torch.save(model.state_dict(), 'mobilenetv3_mamba_wlasl300_state_dict.pth')
    print("State dict saved: mobilenetv3_mamba_wlasl300_state_dict.pth")

# Print final summary
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Model: MobileNetV3-Large + {Config.N_MAMBA_LAYERS}-layer Mamba")
print(f"Dataset: WLASL{Config.NUM_CLASSES} (dual-source fallback loading)")
print(f"Input: {Config.NUM_FRAMES} frames × {Config.FRAME_SIZE}×{Config.FRAME_SIZE} RGB")
print(f"Parameters: {total_params:,}")
print(f"Model Size: {total_params * 4 / (1024**2):.2f} MB")
print(f"Test Top-1: {test_top1:.2f}%")
print(f"Test Top-5: {test_top5:.2f}%")
print(f"Test Top-10: {test_top10:.2f}%")
print(f"GPU Inference: {avg_inference_time:.2f} ms/video")
print(f"CPU Inference: {results_cpu['avg_ms']:.2f} ± {results_cpu['std_ms']:.2f} ms/video")
print("=" * 60)